# Trader 21 - Screening S&P 500 (500 spółek)

Skrypt do automatycznego screeningu 500 spółek z indeksu S&P 500 pod kątem Twoich twardych kryteriów Trader 21.

**Kryteria (sprawdzane):**
- trailingPE < 15
- priceToBook < 1.5
- priceToSalesTrailingTwelveMonths < 1
- enterpriseToEbitda < 10
- returnOnEquity > 0.15
- revenueGrowth > 0.10

Spółki z **≥ 3 spełnionymi kryteriami** są zapisywane do DataFrame i wyświetlane.

**Uwaga:** Uruchamianie może trwać kilka-kilkanaście minut (każde yf.Ticker.info to osobne zapytanie).

In [ ]:
!pip install yfinance pandas tqdm -q
import yfinance as yf
import pandas as pd
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    tables = pd.read_html(url, header=0)
    return tables[0]['Symbol'].tolist()

def trader21_analiza(ticker):
    try:
        t = yf.Ticker(ticker)
        info = t.info
        spelnione = 0
        
        # Twoje twarde kryteria
        if info.get('trailingPE') and info['trailingPE'] < 15: spelnione += 1
        if info.get('priceToBook') and info['priceToBook'] < 1.5: spelnione += 1
        if info.get('priceToSalesTrailingTwelveMonths') and info['priceToSalesTrailingTwelveMonths'] < 1: spelnione += 1
        if info.get('enterpriseToEbitda') and info['enterpriseToEbitda'] < 10: spelnione += 1
        if info.get('returnOnEquity') and info['returnOnEquity'] > 0.15: spelnione += 1
        if info.get('revenueGrowth') and info['revenueGrowth'] > 0.10: spelnione += 1
        
        return {'ticker': ticker, 'spelnione': spelnione}
    except:
        return {'ticker': ticker, 'spelnione': 0}

print("Pobieram S&P 500...")
tickers = get_sp500_tickers()

wyniki = []
for tkr in tqdm(tickers[:500], desc="Sprawdzanie 500 spółek"):
    res = trader21_analiza(tkr)
    if res['spelnione'] >= 3:
        wyniki.append(res)

df = pd.DataFrame(wyniki).sort_values('spelnione', ascending=False)
print(f"\n✓ Znaleziono {len(df)} spółek z ≥3 spełnionymi kryteriami Trader 21")
display(df.head(50))